In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/ML/final project"
RAW_ROOT     = f"{PROJECT_ROOT}/raw_challenge2020"
DATA_DIR     = f"{PROJECT_ROOT}/data"

import os
os.makedirs(RAW_ROOT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT)
print("DATA_DIR:", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/ML/final project
RAW_ROOT: /content/drive/MyDrive/Colab Notebooks/ML/final project/raw_challenge2020
DATA_DIR: /content/drive/MyDrive/Colab Notebooks/ML/final project/data


In [ ]:
%cd "$RAW_ROOT"

# 只下載 Challenge-2020 中的 training/ptb-xl 子資料夾到雲端硬碟
#!wget -r -N -c -np https://physionet.org/files/challenge-2020/1.0.2/training/ptb-xl/

/content/drive/MyDrive/Colab Notebooks/ML/final project/raw_challenge2020


In [ ]:
import glob

ptbxl_paths = glob.glob(f"{RAW_ROOT}/**/training/ptb-xl", recursive=True)
print("Detected ptb-xl paths:", ptbxl_paths)
assert len(ptbxl_paths) > 0, "找不到 training/ptb-xl，確認 wget 有成功跑完。"

PTBXL_DIR = ptbxl_paths[0]
print("Using PTBXL_DIR:", PTBXL_DIR)

Detected ptb-xl paths: ['/content/drive/MyDrive/Colab Notebooks/ML/final project/raw_challenge2020/physionet.org/files/challenge-2020/1.0.2/training/ptb-xl']
Using PTBXL_DIR: /content/drive/MyDrive/Colab Notebooks/ML/final project/raw_challenge2020/physionet.org/files/challenge-2020/1.0.2/training/ptb-xl


In [ ]:
!pip install wfdb neurokit2

import numpy as np
import wfdb
from scipy import signal, io as sio
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import os

# 目標採樣率與長度
TARGET_FS  = 100          # 目標 fs (Hz)
SEG_LEN_S  = 10           # 每筆 10 秒
TARGET_LEN = TARGET_FS * SEG_LEN_S  # 1000 點

N_CHANNELS = 12           # 12 導程 ECG

print("TARGET_FS:", TARGET_FS, "TARGET_LEN:", TARGET_LEN)

TARGET_FS: 100 TARGET_LEN: 1000


In [ ]:
# AF 相關 SNOMED CT 診斷碼
AF_CODES = {"164889003", "164890007"}  # Atrial fibrillation, Atrial flutter

def get_af_label_from_header(header_path):
    """
    從 .hea 讀 SNOMED 診斷碼，看是否包含 AF 相關 code。
    回傳 1 = AF, 0 = non-AF
    """
    with open(header_path, "r") as f:
        lines = f.readlines()

    dx_codes = []
    for line in lines:
        line = line.strip()
        if line.startswith("# Dx"):
            # 例如: "#Dx: 164889003,164890007"
            parts = line.split(":")
            if len(parts) > 1:
                code_str = parts[1].strip()
                dx_codes = [c.strip() for c in code_str.split(",") if c.strip()]
            break

    dx_set = set(dx_codes)
    return 1 if (dx_set & AF_CODES) else 0


def get_fs_from_header(header_path, default_fs=500.0):
    """
    從 header 第一行解析 sampling frequency (fs)。
    格式類似: "A0001 12 500 5000 10-MAY-2019 ..."
    """
    with open(header_path, "r") as f:
        first_line = f.readline().strip()
    parts = first_line.split()
    if len(parts) >= 3:
        try:
            fs = float(parts[2])
            return fs
        except Exception:
            pass
    return default_fs


def bandpass_filter(sig, fs, low=0.5, high=40.0, order=4):
    """
    sig: (T, C)
    Butterworth band-pass filter
    """
    nyq = 0.5 * fs
    b, a = signal.butter(order, [low / nyq, high / nyq], btype="band")
    return signal.filtfilt(b, a, sig, axis=0)


def resample_signal(sig, orig_fs, target_fs):
    """
    sig: (T, C) -> (T_target, C)
    使用 scipy.signal.resample
    """
    T = sig.shape[0]
    T_target = int(round(T * target_fs / orig_fs))
    sig_rs = signal.resample(sig, T_target, axis=0)
    return sig_rs


def normalize_record(sig):
    """
    per-record z-score: (T, C) → (T, C)
    """
    mean = sig.mean(axis=0, keepdims=True)
    std  = sig.std(axis=0, keepdims=True) + 1e-8
    return (sig - mean) / std

In [ ]:
def load_signal(base_path):
    """
    base_path: 不含副檔名，例如 ".../A0001"
    優先讀 .mat，找不到就用 wfdb.rdsamp
    回傳: sig (T, C)
    """
    mat_path = base_path + ".mat"
    if os.path.exists(mat_path):
        m = sio.loadmat(mat_path)
        # Challenge-2020 通常把訊號存在 m['val']，shape: (C, T)
        if "val" in m:
            sig = np.transpose(m["val"])  # (T, C)
        else:
            raise RuntimeError(f"{mat_path} 找不到 'val' 欄位。")
    else:
        # WFDB 格式
        sig, info = wfdb.rdsamp(base_path)  # sig: (T, C)
    return sig


def preprocess_record(header_path):
    """
    從 .hea:
      1) 取得 AF / non-AF label
      2) 找到對應訊號 (mat 或 WFDB)
      3) band-pass / resample / normalize / 固定長度
    回傳:
      x: (12, TARGET_LEN) float32
      y: 0/1 (AF)
    """
    # 1) label
    y = get_af_label_from_header(header_path)
    #if y == 1: print("AF ratio +1 ")

    # 2) base path (去掉 .hea)
    base_path = header_path[:-4]

    # 3) 讀取原始訊號
    sig = load_signal(base_path)  # (T, C)
    T, C = sig.shape

    # 只取前 12 導程，不足就 padding
    if C >= N_CHANNELS:
        sig = sig[:, :N_CHANNELS]
    else:
        pad_w = N_CHANNELS - C
        sig = np.pad(sig, ((0, 0), (0, pad_w)), mode="constant")

    # 4) 取得 fs
    fs = get_fs_from_header(header_path, default_fs=500.0)

    # 5) band-pass
    sig = bandpass_filter(sig, fs=fs)

    # 6) resample 到 TARGET_FS
    sig = resample_signal(sig, orig_fs=fs, target_fs=TARGET_FS)

    # 7) 固定長度處理
    T_rs = sig.shape[0]
    if T_rs < TARGET_LEN:
        pad_len = TARGET_LEN - T_rs
        sig = np.pad(sig, ((0, pad_len), (0, 0)), mode="constant")
    elif T_rs > TARGET_LEN:
        sig = sig[:TARGET_LEN, :]

    # 8) normalize + 轉成 (C, T)
    sig = normalize_record(sig)  # (T, C)
    sig = sig.T                  # (C, T) = (12, TARGET_LEN)

    return sig.astype(np.float32), y

In [ ]:
# 從雲端搬data到colab 裡面
import os, shutil

LOCAL_PTBXL_DIR = "/content/ptbxl_local"

if not os.path.exists(LOCAL_PTBXL_DIR):
    shutil.copytree(PTBXL_DIR, LOCAL_PTBXL_DIR)
    print("Copied PTBXL to local:", LOCAL_PTBXL_DIR)
else:
    print("Local PTBXL already exists:", LOCAL_PTBXL_DIR)


Local PTBXL already exists: /content/ptbxl_local


In [ ]:
import glob

#header_files = glob.glob(os.path.join(PTBXL_DIR, "**", "*.hea"), recursive=True)
header_files = glob.glob(os.path.join(LOCAL_PTBXL_DIR, "**", "*.hea"), recursive=True)

print("Total header files found:", len(header_files))

all_X = []
all_y = []

for h in tqdm(header_files):
    try:
        x, y = preprocess_record(h)
        all_X.append(x)  # (12, TARGET_LEN)
        all_y.append(y)
    except Exception as e:
        print("Error processing", h, ":", e)

all_X = np.stack(all_X, axis=0)           # (N, 12, TARGET_LEN)
all_y = np.array(all_y, dtype=np.int64)   # (N,)

print("all_X shape:", all_X.shape)
print("all_y shape:", all_y.shape)
print("AF ratio:", all_y.mean())

Total header files found: 21843


100%|██████████| 21843/21843 [02:58<00:00, 122.56it/s]


all_X shape: (21843, 12, 1000)
all_y shape: (21843,)
AF ratio: 0.07187657373071464


In [ ]:
import neurokit2 as nk
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

extracted_X = []
extracted_y = []
features = []

for i in tqdm(range(all_X.shape[0])):
    sig = all_X[i][1]  # Index 1 represents "Lead II"
    try:
      processed_sig, info = nk.ecg_process(sig, sampling_rate=TARGET_FS)
      if len(info['ECG_R_Peaks']) < 5:
          print(f"  [Debug] 第{i}筆資料R波數量不足")
          continue

      features = nk.ecg_analyze(processed_sig, sampling_rate=TARGET_FS, method="interval-related")
      features = features.astype(float)
      if features is None or features.empty:
          print(f"  [Debug] 第{i}筆資料取不到feature")
          continue

      extracted_X.append(features)
      extracted_y.append(all_y[i])
    except Exception as e:
        print(f"  [Debug] 第{i}筆資料錯誤: {e}")

print(f"提取出的X筆數: {len(extracted_X)}")
print(f"提取出的y筆數: {len(extracted_y)}")

# 1. 使用 concat 合併，自動對齊所有特徵欄位 (解決 inhomogeneous shape 問題)
extracted_df = pd.concat(extracted_X, ignore_index=True)

# 2. 填補缺值 (因為有些訊號可能算不出某些特徵，產生 NaN)
# SVM 不接受 NaN，這裡補 0 或平均值皆可
extracted_df = extracted_df.fillna(0)

# 3. 轉回 Numpy Array 供模型使用
extracted_X = extracted_df.to_numpy()
extracted_y = np.array(extracted_y)

print("Extracted_X shape:", extracted_X.shape)
print("Extracted_y shape:", extracted_y.shape)

  0%|          | 17/21843 [00:06<2:34:44,  2.35it/s]

  [Debug] 第16筆資料錯誤: All-NaN slice encountered


  0%|          | 25/21843 [00:13<3:16:28,  1.85it/s]

  [Debug] 第25筆資料錯誤: cannot convert float NaN to integer


  1%|▏         | 312/21843 [01:35<1:28:03,  4.08it/s]

  [Debug] 第311筆資料錯誤: All-NaN slice encountered


... (280 outputs left out)

 99%|█████████▉| 21624/21843 [1:14:17<00:32,  6.76it/s]

  [Debug] 第21622筆資料錯誤: cannot convert float NaN to integer


100%|██████████| 21843/21843 [1:15:02<00:00,  4.85it/s]


提取出的X筆數: 21699
提取出的y筆數: 21699
Extracted_X shape: (21699, 84)
Extracted_y shape: (21699,)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    extracted_X, extracted_y,
    test_size=0.2,
    random_state=42,
    stratify=extracted_y
)

print("SVM_X_train shape:", X_train.shape)
print("SVM_y_train shape:", y_train.shape)
print("SVM_X_val shape:", X_val.shape)
print("SVM_y_val shape:", y_val.shape)

np.save(os.path.join(DATA_DIR, "SVM_X_train.npy"), X_train)
np.save(os.path.join(DATA_DIR, "SVM_y_train.npy"), y_train)
np.save(os.path.join(DATA_DIR, "SVM_X_val.npy"),   X_val)
np.save(os.path.join(DATA_DIR, "SVM_y_val.npy"),   y_val)

print("Saved:")
print(os.path.join(DATA_DIR, "SVM_X_train.npy"))
print(os.path.join(DATA_DIR, "SVM_y_train.npy"))
print(os.path.join(DATA_DIR, "SVM_X_val.npy"))
print(os.path.join(DATA_DIR, "SVM_y_val.npy"))

SVM_X_train shape: (17359, 84)
SVM_y_train shape: (17359,)
SVM_X_val shape: (4340, 84)
SVM_y_val shape: (4340,)
Saved:
/content/drive/MyDrive/Colab Notebooks/ML/final project/data/SVM_X_train.npy
/content/drive/MyDrive/Colab Notebooks/ML/final project/data/SVM_y_train.npy
/content/drive/MyDrive/Colab Notebooks/ML/final project/data/SVM_X_val.npy
/content/drive/MyDrive/Colab Notebooks/ML/final project/data/SVM_y_val.npy


In [ ]:
# 確認檔案存在於雲端硬碟
print("Files in DATA_DIR:", os.listdir(DATA_DIR))

Files in DATA_DIR: ['SVM_X_train.npy', 'SVM_y_train.npy', 'SVM_X_val.npy', 'SVM_y_val.npy']
